In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Johnson2020_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["SM006", "SM012", "SM017", "SM018"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 21476 × 19144
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 1476/1476 [00:02<00:00, 666.10it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,446,1.747375,True,0.006512,0.003350,0.521826
AC092667.2,False,270,1.057828,True,0.004544,0.002554,0.562616
ZNF367,False,1279,5.010970,True,0.022542,0.012733,0.543734
SULT1B1,False,93,0.364363,True,0.001705,0.001093,0.721791
TRIM63,False,28,0.109701,True,0.000591,0.000388,0.556369
...,...,...,...,...,...,...,...
CFAP298-TCP10L,False,41,0.160633,True,0.000661,0.000340,0.503573
ARPIN-AP3S2,False,15,0.058768,True,0.000253,0.000135,0.591846
SPATA46,False,16,0.062686,True,0.000357,0.000285,1.011082


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 21476 × 16050
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
# Create a DataFrame with the metadata for Johnson2020
metadata_data = {
    'Author': ['Johnson2020'] * 4,
    'donor_id': ["SM006", "SM012", "SM017", "SM018"],
    'stage': ['Primary'] * 4,
    'assay': ['10x 3\' v3'] * 4,
    'tissue': ['left temporal lobe', 'right parietal lobe', 'right frontal lobe', 'right frontal lobe'],
    'Cells': ['Total'] * 4,
    'Method': ['cell'] * 4
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)


        Author donor_id    stage      assay               tissue  Cells Method
0  Johnson2020    SM006  Primary  10x 3' v3   left temporal lobe  Total   cell
1  Johnson2020    SM012  Primary  10x 3' v3  right parietal lobe  Total   cell
2  Johnson2020    SM017  Primary  10x 3' v3   right frontal lobe  Total   cell
3  Johnson2020    SM018  Primary  10x 3' v3   right frontal lobe  Total   cell


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id       Author    stage      assay              tissue  Cells  \
0        SM006  Johnson2020  Primary  10x 3' v3  left temporal lobe  Total   
1        SM006  Johnson2020  Primary  10x 3' v3  left temporal lobe  Total   
2        SM006  Johnson2020  Primary  10x 3' v3  left temporal lobe  Total   
3        SM006  Johnson2020  Primary  10x 3' v3  left temporal lobe  Total   
4        SM006  Johnson2020  Primary  10x 3' v3  left temporal lobe  Total   
...        ...          ...      ...        ...                 ...    ...   
21471    SM018  Johnson2020  Primary  10x 3' v3  right frontal lobe  Total   
21472    SM018  Johnson2020  Primary  10x 3' v3  right frontal lobe  Total   
21473    SM018  Johnson2020  Primary  10x 3' v3  right frontal lobe  Total   
21474    SM018  Johnson2020  Primary  10x 3' v3  right frontal lobe  Total   
21475    SM018  Johnson2020  Primary  10x 3' v3  right frontal lobe  Total   

      Method  
0       cell  
1       cell  
2       cell  
3  

In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
SM006_AAACCCAAGTCTTCGA-1-0,SM006,3194,2303.947998,Neoplastic,Differentiated-like,AC-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
SM006_AAACCCAGTACGGGAT-1-0,SM006,2379,1708.597778,Neoplastic,Differentiated-like,AC-like,Astrocyte,Radial Glia Cells,malignant cell
SM006_AAACCCATCACTGTCC-1-0,SM006,4004,2239.699219,Neoplastic,Differentiated-like,AC-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
SM006_AAACCCATCATCTCTA-1-0,SM006,4061,1980.843262,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
SM006_AAACGAAAGGACATCG-1-0,SM006,2910,1711.328369,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
...,...,...,...,...,...,...,...,...,...
SM018_TTTGGAGCATCCTAAG-1-0,SM018,4517,1714.838013,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
SM018_TTTGGAGGTACCTGTA-1-0,SM018,6071,1879.358276,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
SM018_TTTGGTTCATACAGCT-1-0,SM018,3673,1630.754395,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
SM018_TTTGGTTTCCGATCTC-1-0,SM018,2216,2007.818726,Neoplastic,Stem-like,OPC-like,Astrocyte,Radial Glia Cells,malignant cell


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    687 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    729 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Johnson2020_Part3.h5ad")